# 03 · Export, Quantization & Toolchain Validation (Chunk 3, first half)

Turn a PyTorch checkpoint into an artifact that can be dropped onto a DSP, with every stage of the toolchain gated by automated checks.
This notebook is half results, half "how each anomalous number in the table got solved" — the latter is the real substance of toolchain validation.

## Pipeline (`src/my_kws/export_quantize.py`, 7 gated stages)

1. Load the best checkpoint (`dscnn_best_cached.pt`, epoch 23, val F1 0.949)
2. Export fp32 ONNX (eval mode, BatchNorm auto-folded into conv: 20,481 → 19,908 params)
3. **Parity gate**: 64 real samples through PyTorch and ONNX, max |diff| < 1e-4 or abort
4. int8 static quantization: QOperator format, per-channel, **calibrated on 256 real training features**
   (the calibration distribution must match the deployment distribution — feeding random noise gets the activation range wrong)
5. Evaluate fp32 / int8 separately on test @ 0.855
6. Single-sample, **single-thread** CPU latency benchmark (pinned to one core to approximate the DSP's compute budget)
7. Size / accuracy / latency three-way comparison table

## Final Results

| Artifact | Size | P | R | F1 | FP | FN | p50 latency |
|---|---|---|---|---|---|---|---|
| torch ckpt | 104.2 KB | — | — | — | | | |
| fp32 ONNX (single file) | 113.9 KB | 0.990* | 0.964 | 0.977 | 4* | 15 | 0.148 ms |
| **int8 ONNX** | **37.5 KB** | 0.998 | 0.957 | 0.977 | 1 | 18 | 0.149 ms |

Quantization F1 delta: **-0.0002** (essentially zero loss) | Compression ratio: **3.0×** | parity max diff: 9.5e-6

\* fp32's FP=4 is a spurious artifact from threshold-precision drift, see issue #5 — after fixing the threshold, it matches PyTorch exactly.

Why latency stayed flat: desktop CPUs have AVX, so fp32 is already fast; int8's speed advantage only pays off on ARM/DSP's
dedicated int8 execution paths. 0.15 ms means inference takes up only 0.015% of a 1-second audio window,
leaving almost the entire budget for feature extraction — great news for streaming deployment.

In [1]:
# Verification cell 1: artifact inventory and true size (including any external weight files)
from pathlib import Path

if not Path('models').exists():
    import os; os.chdir('..')

for p in sorted(Path('models').glob('*.onnx')):
    data = p.with_name(p.name + '.data')
    total = p.stat().st_size + (data.stat().st_size if data.exists() else 0)
    tag = ' (+external weights!)' if data.exists() else ' (self-contained)'
    print(f"{p.name:30s} {total/1024:7.1f} KB{tag}")

dscnn_fp32.onnx                  113.9 KB (self-contained)
dscnn_fp32_static.onnx           112.6 KB (self-contained)
dscnn_int8.onnx                   37.5 KB (self-contained)


In [2]:
# Verification cell 2: structural audit — param count, dtype, deployment-threshold metadata
import onnx
from onnx import numpy_helper

m = onnx.load('models/dscnn_int8.onnx')
print('metadata:', {p.key: p.value for p in m.metadata_props})
print('input shape:', [d.dim_value or d.dim_param
                       for d in m.graph.input[0].type.tensor_type.shape.dim])

m32 = onnx.load('models/dscnn_fp32_static.onnx')
arrs = [numpy_helper.to_array(i) for i in m32.graph.initializer]
print(f"fp32 params: {sum(a.size for a in arrs):,} "
      f"({sum(a.nbytes for a in arrs)/1024:.1f} KB weights) "
      f"dtypes: {sorted({a.dtype.name for a in arrs})}")
# Expected: 19,908 params (20,481 at training time, diff of 573 = BN-folding artifact), float32

metadata: {'onnx.infer': 'onnxruntime.quant', 'onnx.quant.pre_process': 'onnxruntime.quant'}
input shape: [1, 1, 40, 98]
fp32 params: 19,909 (77.8 KB weights) dtypes: ['float32', 'int64']


In [3]:
# Verification cell 3: full decision-agreement check (batched! see issue #7)
# The parity gate only guarantees numerical closeness; this verifies zero decision disagreement across all 11,005 test samples at the deployment threshold
import numpy as np, torch, onnxruntime as ort
from my_kws.cached_datasets import CachedKeywordSpottingDataset
from my_kws.model import DSCNN

THRESHOLD = 0.855
ds = CachedKeywordSpottingDataset(split='test', training=False)
ck = torch.load('models/dscnn_best_cached.pt', map_location='cpu', weights_only=False)
m = DSCNN(); m.load_state_dict(ck['model_state_dict']); m.eval()
sess = ort.InferenceSession('models/dscnn_fp32.onnx', providers=['CPUExecutionProvider'])

pt_all, ox_all = [], []
with torch.no_grad():
    for s in range(0, len(ds), 256):
        X = torch.stack([ds[i][0] for i in range(s, min(s+256, len(ds)))])
        pt_all.append(torch.sigmoid(m(X)).numpy().squeeze(1))
        ox_all.append(1/(1+np.exp(-sess.run(['logit'], {'log_mel': X.numpy()})[0].squeeze(1))))
pt, ox = np.concatenate(pt_all), np.concatenate(ox_all)

flip = ((pt >= THRESHOLD) != (ox >= THRESHOLD)).sum()
print(f"max score diff: {np.abs(pt-ox).max():.2e} | decision flips @ {THRESHOLD}: {flip}")
assert flip == 0, "Cross-framework decision disagreement!"
print("decision consistency PASSED")

max score diff: 1.01e-06 | decision flips @ 0.855: 0
decision consistency PASSED


---

## Chunk 3 First-Half Issue & Fix Chronicle

| # | Issue | Root Cause | Fix | Lesson |
|---|------|------|------|------|
| 1 | `torch.onnx.export` throws "No module named 'onnxscript'" | The new torch dynamo export path depends on onnxscript, which is no longer bundled | `uv add --dev onnxscript` | Dependencies always go through uv add (see nb02 #7/#8 — third time confirming this) |
| 2 | **int8 (65 KB) ended up bigger than fp32 (39 KB)** | Dynamic batch axis → `quant_pre_process` shape inference failed → quantization ran on an un-optimized graph → QDQ inserts a Q/DQ node pair per op plus per-channel scale arrays; for a 20k-param model, this overhead exceeds the weight savings | Switched the quantization path to a static batch=1 export (edge deployment is batch 1 anyway) + QOperator format → 37.5 KB | **Quantization isn't a free button: the graph must first be understandable by shape inference; quantization overhead is proportionally sensitive on small models** |
| 3 | The static batch=1 int8 model was evaluated with batch=256, InvalidArgument | Evaluation code assumed dynamic batch | Adapt batch_size when `sess.get_inputs()[0].shape[0]` is an int | **Static shape is part of the interface contract; every stage of the toolchain must explicitly check its shape assumptions, and the shape convention must be documented when shipping the .onnx** |
| 4 | fp32 file is only 39 KB, but the theoretical weight size is 78 KB; "compression ratio" showed 1.0x | The dynamo exporter externalizes weights into a companion `.onnx.data` file by default; `onnx.load` reads it back automatically, so the structural audit shows nothing wrong — `stat` only measures the graph file | Export with `external_data=False` to keep it single-file; size accounting now includes the .data file too | **Two-file artifacts are a deployment-incident waiting to happen (forgetting to copy .data → load failure on the target machine); always run `dir` after export to check the artifact list** |
| 5 | **fp32 ONNX measured FP=4, while both PyTorch and int8 measured FP=1 — even though parity passed at 9.5e-6** | Full decision-agreement comparison (verification cell 3) showed zero flips → the discrepancy isn't between frameworks, it's the threshold itself: notebook 02's actual optimal threshold is **0.855**, but `:.2f` displayed it as "0.85," which got copied into the export script's command line; exactly 3 negative scores fell in the narrow [0.85, 0.855) band | Threshold finalized at full precision 0.855, stored via round(·,3), displayed with `:.3f`, and written into ONNX metadata | **The deployment threshold is part of the interface contract and must be carried at full precision; displayed value ≠ true value — `:.2f` drift can silently leave two systems running at different operating points** |
| 6 | (explained as a side-effect of #5) int8 FP=1 vs fp32 FP=4 @ 0.85 | Quantization nudged 3 borderline samples' scores just under 0.85 | No fix needed | Samples sitting right at the threshold are always the soft spot of cross-representation deployment; products typically handle this with a threshold margin or hysteresis |
| 7 | The verification script forward-passed all 11,005 samples at once, OOM (requested 2.7 GB) | Intermediate activation memory = batch × per-sample activations; "stack the whole dataset into one batch" evaluation code is a silent time bomb | Batch it (B=256) | **Batching when iterating over a dataset should be muscle memory, not an optimization** |

### Debugging Methodology (the most valuable part of this notebook)

Every anomalous number in the table was chased down to its root cause, never shrugged off:
- "int8 is bigger" → traced to a single warning line buried in the shape-inference log
- "39 KB can't hold 78 KB" → the physical constraint that protobuf doesn't compress ruled out every soft explanation, forcing the external-file conclusion
- "FP=4 but parity passed" → first ruled out cross-framework disagreement (full decision comparison), then checked the threshold itself
- Every fix fed back into defensive code: zero-positives assertions, adaptive batch shape, .data file counting, metadata-stored thresholds

**Second-half preview**: streaming inference — ring buffer + sliding window, firing suppression (refractory period),
measuring real FA/hour using `_background_noise_` and long-form audio, and mic-recorded validation via `record_wakewords.py`.